In [1]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IMDB Dataset.csv"

# Load the latest version
dataset = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", dataset.head())

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
First 5 records:                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [2]:
import torch
import torch.nn as nn
import nltk
nltk.download("punkt_tab")
import re
from nltk.tokenize import word_tokenize



[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [3]:
X = dataset["review"]
y = dataset["sentiment"]

In [4]:
y.unique()
y = y.map({"positive" : 1, "negative" : 0})

In [5]:
def preprocessing(text):
  lower = text.lower()
  base = re.sub(r'[^a-zA-Z0-9]\s', '', lower)
  tokenization = word_tokenize(base)

  return tokenization


In [6]:
X = X.apply(preprocessing)

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


tokenizer = Tokenizer(oov_token="<UNK>", num_words = 30000)

tokenizer.fit_on_texts(X_train)

X_train = tokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(X_train, padding = "post")
X_test =  tokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(X_test, maxlen = X_train.shape[1],  padding = "post")

vocab = tokenizer.word_index


In [9]:
print(max(max(seq) for seq in X_train))

29999


In [10]:
X_train_tensor = torch.tensor(X_train, dtype = torch.long)
X_test_tensor = torch.tensor(X_test, dtype = torch.long)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype = torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype = torch.long)

In [11]:
X_train_tensor

tensor([[  13,   15,   58,  ...,    0,    0,    0],
        [  17,   78,   31,  ...,    0,    0,    0],
        [   3, 1412,  126,  ...,    0,    0,    0],
        ...,
        [ 962,   16, 2938,  ...,    0,    0,    0],
        [  16,  382,   18,  ...,    0,    0,    0],
        [  16,   11,    3,  ...,    0,    0,    0]])

In [ ]:
embedding = nn.Embedding(num_embeddings = len(vocab) + 1, embedding_dim = 128)

X_train_embedded = embedding(X_train_tensor)
X_test_embedded = embedding(X_test_tensor)

print(X_train_embedded.shape)
print(X_test_embedded.shape)

In [ ]:
print("Vocabulary Size:", len(vocab))
print("Maximum Sequence Length:", X_train_tensor.shape[1])
print("Number of Classes:", len(set(y_train)))
print("y_train dtype:", y_train_tensor.dtype)

In [ ]:
class Transformer(nn.Module):

  def __init__(self, embedding_dim = 128, max_length = 256, num_heads = 8, num_layers = 4, num_classes = 2, dropout = 0.1, dim_feedforward = 512):

    super().__init__()

    self.postinal_embedding = nn.Embedding(max_length, embedding_dim)

    encoder_layer = nn.TransformerEncoderLayer(d_model = embedding_dim, nhead = num_heads, dropout = dropout, dim_feedforward = dim_feedforward, batch_first = True)

    self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)

    self.dropout = nn.Dropout(dropout)

    self.fc = nn.Linear(embedding_dim, num_classes)

  def forward(self, x):

    batch_size, seq_len, embedding_dim = x.shape
    positions = torch.arange(seq_len, device = x.device).unsqueeze(0).expand(batch_size, seq_len)

    print("x shape:", x.shape)
    print("seq_len:", seq_len)
    print("max_length:", self.positional_embedding.num_embeddings)

    x = x + self.postinal_embedding(positions)

    x = self.encoder(x)

    x = x.mean(dim = 1)

    x = self.dropout(x)

    return self.fc(x)


In [ ]:
model = Transformer()
outputs = model(X_train_embedded)

In [ ]:
train_encoding = tokenizer(
    X_train.tolist(),
    padding=True,
    truncation=True,
    max_length=256
)

test_encoding = tokenizer(
    X_test.tolist(),
    padding=True,
    truncation=True,
    max_length=256
)

In [ ]:
train_encoding = tokenize_reviews(X_train)
test_encoding = tokenize_reviews(X_test)

In [ ]:
train_encoding[3]

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class CustomDataset(Dataset):

  def __init__(self, encoding, label):

    self.encoding = encoding
    self.label = label

  def __len__(self):

    return len(self.label)

  def __getitem__(self, idx):

    item = {}

    for key, value in self.encoding.items():
      item[key] = torch.tensor(value[idx])

    item["label"] = torch.tensor(self.label[idx])

    return item

In [ ]:
train_dataset = CustomDataset(train_encoding, y_train.tolist())
test_dataset = CustomDataset(test_encoding, y_test.tolist())

In [ ]:
train_dataset

In [ ]:
# Loader

train_loader = DataLoader(train_dataset, batch_size = 100, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 100)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

In [ ]:
model.train()

epochs = 3

for epoch in range(epochs):

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: {total_loss/len(train_loader):.4f}")

In [ ]:
print(type(train_encoding))
print(type(test_encoding))
print(type(train_encoding[0]))
